# Statistical Drivers of Loan Default Risk

---

## Executive Summary

This notebook presents a consolidated, executive-ready analysis of the statistical drivers of loan default risk across a diversified lending portfolio of approximately **1,000 loans**.

The analysis moves beyond descriptive averages to identify **where risk is concentrated**, **how uncertainty alters risk perception**, and **which loans require immediate action**. By integrating default probabilities, confidence intervals, exposure concentration, stress scenarios, and early warning indicators, the framework supports proactive credit risk management rather than reactive loss reporting.

Key conclusions show that:
- Risk is **unevenly distributed across products**
- **Tail exposure** materially amplifies loss severity
- Confidence-adjusted risk bands significantly change prioritization
- Over **25% of the portfolio requires heightened monitoring or escalation**

This notebook represents the **final synthesis** of the analytical workflow and is designed for executive review and interview walkthroughs.

---

## 1. Data Overview & Quality Controls

### Dataset Scope
- **Total loans:** ~1,000  
- **Loan products:** Mortgage, Home Equity, Personal Loan, Auto Loan, Business Loan  
- **Key variables:**  
  - Behavioural: payments missed, delinquency flag, NPL flag  
  - Exposure: loan amount, outstanding balance  
  - Structural: loan type, term, interest rate  

### Data Quality Controls Applied
Before analysis:
- Loan product names were **standardized** to remove case and formatting inconsistencies  
- Monetary fields were validated to ensure **numeric integrity**  
- Default and delinquency indicators were reviewed for logical consistency  

These steps ensured that risk metrics reflect **true portfolio behaviour**, not data artefacts.

---

---

## 1. Data Overview & Quality Controls

### Dataset Scope
- **Total loans:** ~1,000  
- **Loan products:** Mortgage, Home Equity, Personal Loan, Auto Loan, Business Loan  
- **Key variables:**  
  - Behavioural: payments missed, delinquency flag, NPL flag  
  - Exposure: loan amount, outstanding balance  
  - Structural: loan type, term, interest rate  

### Data Quality Controls Applied
Before analysis:
- Loan product names were **standardized** to remove case and formatting inconsistencies  
- Monetary fields were validated to ensure **numeric integrity**  
- Default and delinquency indicators were reviewed for logical consistency  

These steps ensured that risk metrics reflect **true portfolio behaviour**, not data artefacts.

---

## Data Loading & Preparation

This section loads the loan dataset and applies the same cleaning and feature
engineering steps used in the analysis notebook to ensure consistency and
reproducibility of results.


### Environment Setup & Data Loading

In [2]:
import pandas as pd
import numpy as np

# Load loans dataset (local path)
df = pd.read_csv(
    r"C:\Users\princ\OneDrive\Desktop\PROJECTS\loan-performance-analytics\data\loans.csv"
)

# Quick validation
df.shape


(1000, 19)

1) Full dependency/import cell

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# Day 5 (chi-square)
from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)


2) Load + standardize types (fix “strings to values” + product name cleanup)

In [4]:
# Load
df = pd.read_csv(r"C:\Users\princ\OneDrive\Desktop\PROJECTS\loan-performance-analytics\data\loans.csv")

# --- Helper: convert messy string flags to 0/1 safely ---
def to_binary_flag(s: pd.Series) -> pd.Series:
    """
    Converts common string/boolean/numeric representations into 0/1.
    Handles: Yes/No, Y/N, True/False, T/F, 1/0, '1'/'0', etc.
    Unknowns -> NaN (then we can fill or leave as missing)
    """
    if s is None:
        return s
    
    # If already numeric 0/1, keep it
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)

    x = s.astype(str).str.strip().str.lower()

    mapping = {
        "1": 1, "0": 0,
        "true": 1, "false": 0,
        "t": 1, "f": 0,
        "yes": 1, "no": 0,
        "y": 1, "n": 0,
        "default": 1, "non-default": 0,   # optional, just in case
    }
    out = x.map(mapping)

    # if some values were numeric-but-string like "1.0"
    out = out.fillna(pd.to_numeric(x, errors="coerce"))

    return out.astype(float)


# --- Normalize column names (optional but helpful) ---
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_")
)

# --- Clean product name casing/spaces (your Day 3 issue) ---
if "loan_type" in df.columns:
    df["loan_type_clean"] = (
        df["loan_type"]
        .astype(str)
        .str.strip()
        .str.title()
    )
else:
    # if your dataset already uses loan_type_clean, keep it consistent
    df["loan_type_clean"] = df.get("loan_type_clean", pd.Series(["Unknown"] * len(df)))

# --- Coerce numeric fields that might be strings (common breaking point) ---
numeric_cols = [
    "loan_amount",
    "outstanding_balance",
    "interest_rate",
    "term_months",
    "payments_missed"
]
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Convert flags that might be strings into numeric 0/1 ---
flag_cols = ["default_flag", "delinquent_flag", "npl_flag"]
for c in flag_cols:
    if c in df.columns:
        df[c] = to_binary_flag(df[c])

# --- If delinquent/npl not provided, recreate from payments_missed (your earlier logic) ---
if "delinquent_flag" not in df.columns and "payments_missed" in df.columns:
    df["delinquent_flag"] = (df["payments_missed"] >= 1).astype(int)

if "npl_flag" not in df.columns and "payments_missed" in df.columns:
    df["npl_flag"] = (df["payments_missed"] >= 3).astype(int)

# default_flag: if missing but you have another column, adapt here
# df["default_flag"] = ...

# --- Final: ensure flags are clean ints (0/1) where possible ---
for c in ["default_flag", "delinquent_flag", "npl_flag"]:
    if c in df.columns:
        df[c] = df[c].fillna(0).astype(int)

# Quick sanity checks
df.shape, df[["loan_type_clean"]].head()


((1000, 22),
   loan_type_clean
 0   Personal Loan
 1       Auto Loan
 2   Personal Loan
 3   Personal Loan
 4        Mortgage)

3) Quick “did we actually fix it?” validation cell

In [5]:
# Confirm flags are truly numeric 0/1
check_cols = [c for c in ["default_flag", "delinquent_flag", "npl_flag"] if c in df.columns]
df[check_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
delinquent_flag,1000.0,0.412,0.492441,0.0,0.0,0.0,1.0,1.0
npl_flag,1000.0,0.262,0.439943,0.0,0.0,0.0,1.0,1.0


In [6]:
# Confirm loan_type is harmonized (no duplicates like "Mortgage" vs "mortgage")
df["loan_type_clean"].value_counts()


loan_type_clean
Auto Loan        274
Mortgage         246
Personal Loan    235
Business Loan    138
Home Equity      107
Name: count, dtype: int64

In [9]:
df['loan_amount'].dtype


dtype('float64')

In [10]:
df['loan_amount'].head()


0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: loan_amount, dtype: float64

In [11]:
df['loan_amount'].isna().mean(), df['loan_amount'].isna().sum(), len(df)


(np.float64(1.0), np.int64(1000), 1000)

In [12]:
df.columns.tolist()


['loan_id',
 'account_id',
 'customer_id',
 'loan_type',
 'loan_amount',
 'interest_rate',
 'term_months',
 'origination_date',
 'maturity_date',
 'loan_status',
 'outstanding_balance',
 'monthly_payment',
 'payments_made',
 'payments_missed',
 'last_payment_date',
 'next_payment_due',
 'collateral_value',
 'credit_score_at_origination',
 'loan_officer_id',
 'loan_type_clean',
 'delinquent_flag',
 'npl_flag']